heading pre processing pipeline for the svm

sub heading : about the data 

we are using the BCI Competition IV: data set 2a specifically in for the prupose of this project this data was aqiured from a study conducted by (cite refrenceBCI Competition 2008– Graz data set A
C. Brunner1, R. Leeb1, G. R. M¨uller-Putz1, A. Schl¨ogl2, and G.
Pfurtscheller1
1Institute for Knowledge Discovery, Graz University of Technology,
Austria
2Institute for Human-Computer Interfaces, Graz University of
Technology, Austria)

loading the required libraries 

In [3]:
import os
import numpy as np
import mne
from mne.preprocessing import ICA

filtering 

In [4]:

# Suppress non-critical warnings to keep the console clean during batch processing
mne.set_log_level('ERROR')

# ==========================================
# Directory Setup
# ==========================================
input_dir = "raweegdata"
output_dir = "final_clean_eeg"
os.makedirs(output_dir, exist_ok=True)

# BCI Competition IV 2a: 9 subjects, 2 sessions each (Training & Evaluation)
subjects = [f"A0{i}" for i in range(1, 10)]
sessions = ["T", "E"]

print("Starting Continuous Preprocessing Pipeline...")
print("=" * 45)

for sub in subjects:
    for sess in sessions:
        filename = f"{sub}{sess}.gdf"
        filepath = os.path.join(input_dir, filename)
        
        if os.path.exists(filepath):
            print(f"\nProcessing {filename}...")
            
            # 1. Load Continuous Data
            raw = mne.io.read_raw_gdf(filepath, preload=True)
            # Map the correct channel types (EEG vs. EOG)
            raw.set_channel_types({'EOG-left': 'eog', 'EOG-central': 'eog', 'EOG-right': 'eog'})
            
            # 2. High-Pass Filter (1.0 Hz)
            # Stabilizes the baseline strictly to feed a clean, drift-free signal into the ICA algorithm.
            print("  -> Applying 1.0 Hz high-pass filter for ICA stability...")
            raw.filter(l_freq=1.0, h_freq=None, fir_design='firwin')
            
            # 3. Fit & Apply ICA
            print("  -> Fitting ICA (Rank-reduced component selection)...")
            # Reverting to 15 components forces the algorithm to consolidate artifacts 
            # into distinct unmixing matrices, preventing variance fragmentation.
            ica = ICA(n_components=15, max_iter='auto', random_state=97)
            ica.fit(raw)
            
            # Automated Artifact Rejection
            print("  -> Hunting for vertical and horizontal EOG artifacts...")
            # By leaving ch_name blank, MNE automatically cross-correlates against 
            # all channels we previously mapped as 'eog' (Central, Left, Right).
            eog_indices, eog_scores = ica.find_bads_eog(raw)
            
            if eog_indices:
                # Deduplicate indices just in case multiple EOG channels flag the same component
                unique_eog_indices = list(set(eog_indices))
                print(f"  -> Found EOG component(s): {unique_eog_indices}. Removing...")
                ica.exclude = unique_eog_indices
            else:
                print("  -> No EOG found (Check data if this happens).")
                
            # Subtract the blinks/saccades from the continuous stream
            ica.apply(raw)
            
            # 4. Bandpass Filter (8–30 Hz)
            # Apply target SMR filter across the clean, continuous stream to eliminate edge artifacts.
            print("  -> Applying continuous 8-30 Hz Motor Imagery filter...")
            raw.filter(l_freq=8.0, h_freq=30.0, fir_design='firwin')
            
            # 5. Apply Common Average Reference (CAR)
            # Re-reference to the global average to improve spatial resolution.
            print("  -> Applying Common Average Reference (CAR)...")
            raw.set_eeg_reference(ref_channels='average')
            
            # 6. Save as .fif and Clear Memory
            # Export the fully processed file (with all embedded event triggers)
            out_filename = f"{sub}{sess}_clean.fif"
            out_filepath = os.path.join(output_dir, out_filename)
            raw.save(out_filepath, overwrite=True)
            print(f"  -> ✅ Saved fully processed file: {out_filename}")
            
            # Free up RAM for the next iteration
            del raw, ica
            
        else:
            print(f"\n❌ Missing: {filename}")

print("\n" + "=" * 45)
print("Pipeline Complete. Continuous data is ready for feature extraction/epoching.")
print("=" * 45)

Starting Continuous Preprocessing Pipeline...

Processing A01T.gdf...
  -> Applying 1.0 Hz high-pass filter for ICA stability...
  -> Fitting ICA (Rank-reduced component selection)...
  -> Hunting for vertical and horizontal EOG artifacts...
  -> Found EOG component(s): [np.int64(0), np.int64(1), np.int64(6)]. Removing...
  -> Applying continuous 8-30 Hz Motor Imagery filter...
  -> Applying Common Average Reference (CAR)...
  -> ✅ Saved fully processed file: A01T_clean.fif

Processing A01E.gdf...
  -> Applying 1.0 Hz high-pass filter for ICA stability...
  -> Fitting ICA (Rank-reduced component selection)...
  -> Hunting for vertical and horizontal EOG artifacts...
  -> Found EOG component(s): [np.int64(0), np.int64(1), np.int64(6)]. Removing...
  -> Applying continuous 8-30 Hz Motor Imagery filter...
  -> Applying Common Average Reference (CAR)...
  -> ✅ Saved fully processed file: A01E_clean.fif

Processing A02T.gdf...
  -> Applying 1.0 Hz high-pass filter for ICA stability...
  -> 

In [ ]:
ephoching

In [6]:
import os
import mne
import numpy as np

# Suppress non-critical warnings to keep the console clean during batch processing
mne.set_log_level('ERROR')

print("Starting Epoching and Artifact Scrubbing Pipeline...")
print("=" * 60)

# ==========================================
# Step 1: Initialization & Directory Mapping
# ==========================================
input_dir = "final_clean_eeg"
output_dir = "epoched_eeg_data"
os.makedirs(output_dir, exist_ok=True)

# Generate list of all files ending in _clean.fif
clean_files = [f for f in os.listdir(input_dir) if f.endswith('_clean.fif')]

if not clean_files:
    print(f"❌ No '_clean.fif' files found in '{input_dir}'.")
else:
    # ==========================================
    # Step 2: Class Dictionary Definition
    # ==========================================
    # Mapping BCI Comp IV 2a string annotations to standard integers (1 to 4)
    motor_imagery_mapping = {
        '769': 1,  # Left Hand
        '770': 2,  # Right Hand
        '771': 3,  # Both Feet
        '772': 4   # Tongue
    }

    # ==========================================
    # Step 3: Batch Loop & Continuous Load
    # ==========================================
    for filename in clean_files:
        filepath = os.path.join(input_dir, filename)
        print(f"\nProcessing {filename}...")
        
        # Instantiate the 25-channel continuous data as the 'raw' variable in memory
        raw = mne.io.read_raw_fif(filepath, preload=True)
        
        # ==========================================
        # Step 4: Event Extraction & Artifact Scrubbing
        # ==========================================
        # Extract all events to find what markers actually exist in this specific file
        all_events, all_event_dict = mne.events_from_annotations(raw)
        
        # Artifact handling
        if '1023' in all_event_dict:
            artifact_id = all_event_dict['1023']
            artifact_samples = all_events[all_events[:, 2] == artifact_id, 0]
            print(f"  -> Found {len(artifact_samples)} expert-flagged artifact markers (Code 1023).")
        else:
            artifact_samples = []
            print("  -> No expert-flagged artifacts found in this session.")
            
        # ------------------------------------------
        # NEW LOGIC: Training vs. Evaluation Routing
        # ------------------------------------------
        # Check if this is an Evaluation file (labels hidden as '783')
        if '783' in all_event_dict and not any(k in all_event_dict for k in ['769', '770', '771', '772']):
            print("  -> Evaluation set detected. True labels are hidden. Mapping cues to '0' (Unknown).")
            dynamic_mapping = {'783': 0}
        else:
            print("  -> Training set detected. Mapping standard 4-class cues.")
            # Only keep the keys that actually exist in this specific file to prevent MNE crashes
            dynamic_mapping = {k: v for k, v in motor_imagery_mapping.items() if k in all_event_dict}
            
        if not dynamic_mapping:
            print("  -> ⚠️ No motor imagery cues found at all. Skipping file.")
            del raw
            continue

        # Extract strictly the motor imagery events using our dynamic mapping
        mi_events, mi_event_dict = mne.events_from_annotations(raw, event_id=dynamic_mapping)
        
        # Filter out any motor imagery event that occurs near a 1023 artifact flag
        clean_mi_events = []
        for event in mi_events:
            event_sample = event[0]
            is_artifact = any(abs(art_sample - event_sample) <= 1000 for art_sample in artifact_samples)
            if not is_artifact:
                clean_mi_events.append(event)
                
        clean_mi_events = np.array(clean_mi_events)
        dropped_count = len(mi_events) - len(clean_mi_events)
        print(f"  -> Purged {dropped_count} bad trials. {len(clean_mi_events)} clean trials remaining.")
        
        if len(clean_mi_events) == 0:
            print("  -> ⚠️ No clean trials remaining. Skipping epoching for this file.")
            del raw
            continue
# Step 5: Epoch Generation & Baselining
        # ==========================================
        print("  -> Slicing continuous data into epochs (0.5s to 3.5s)...")
        epochs = mne.Epochs(
            raw, 
            clean_mi_events, 
            event_id=mi_event_dict,
            tmin=0.5,  # <--- FIXED
            tmax=3.5,  # <--- FIXED
            baseline=None, 
            preload=True
        )
        # ==========================================
        # Step 6: Export & Memory Flush
        # ==========================================
        out_filename = filename.replace('_clean.fif', '_clean-epo.fif')
        out_filepath = os.path.join(output_dir, out_filename)
        
        epochs.save(out_filepath, overwrite=True)
        print(f"  -> ✅ Saved 3D epoch tensor to: {out_filename}")
        
        # Explicit garbage collection to prevent system crashes
        del raw, epochs

print("\n" + "=" * 60)
print("Batch Epoching Complete. Data is shaped and ready for CSP feature extraction.")
print("=" * 60)

Starting Epoching and Artifact Scrubbing Pipeline...

Processing A01E_clean.fif...
  -> Found 7 expert-flagged artifact markers (Code 1023).
  -> Evaluation set detected. True labels are hidden. Mapping cues to '0' (Unknown).
  -> Purged 7 bad trials. 281 clean trials remaining.
  -> Slicing continuous data into epochs (0.5s to 3.5s)...
  -> ✅ Saved 3D epoch tensor to: A01E_clean-epo.fif

Processing A01T_clean.fif...
  -> Found 15 expert-flagged artifact markers (Code 1023).
  -> Training set detected. Mapping standard 4-class cues.
  -> Purged 15 bad trials. 273 clean trials remaining.
  -> Slicing continuous data into epochs (0.5s to 3.5s)...
  -> ✅ Saved 3D epoch tensor to: A01T_clean-epo.fif

Processing A02E_clean.fif...
  -> Found 5 expert-flagged artifact markers (Code 1023).
  -> Evaluation set detected. True labels are hidden. Mapping cues to '0' (Unknown).
  -> Purged 5 bad trials. 283 clean trials remaining.
  -> Slicing continuous data into epochs (0.5s to 3.5s)...
  -> ✅ Sa

csp plus svm bundle 

In [8]:
import os
import mne
import numpy as np

# Suppress non-critical warnings to keep the console clean during batch processing
mne.set_log_level('ERROR')

print("Starting Multi-Subject Data Loader (LOSO Preparation)...")
print("=" * 60)

# ==========================================
# 1. Initialization
# ==========================================
data_dir = 'epoched_eeg_data'
X_all = []
y_all = []
groups_all = []

# ==========================================
# 2. File Hunting
# ==========================================
# Find and sort files to ensure subjects are loaded in numerical order
try:
    epoch_files = sorted([f for f in os.listdir(data_dir) if f.endswith('_clean-epo.fif')])
except FileNotFoundError:
    raise FileNotFoundError(f"❌ Directory '{data_dir}' not found. Ensure previous cells ran successfully.")

if not epoch_files:
    raise ValueError(f"❌ No '_clean-epo.fif' files found in '{data_dir}'.")

# ==========================================
# 3. The Batch Loop
# ==========================================
for filename in epoch_files:
    
    # Extract the subject ID from the filename (e.g., 'A01T_clean-epo.fif' -> 1)
    try:
        subject_id = int(filename[1:3])
    except ValueError:
        print(f"  -> ⚠️ Skipping {filename}: Could not parse integer subject ID.")
        continue

    filepath = os.path.join(data_dir, filename)
    
    # Load the epochs into memory
    epochs = mne.read_epochs(filepath, preload=True)
    
    # Crop to active-state (Crucial for isolating Event-Related Desynchronization)
    epochs.crop(tmin=0.5, tmax=3.5)
    
    # Extract temporary labels to run the validation check
    y_temp = epochs.events[:, 2]
    unique_classes = np.unique(y_temp)
    
    # Validation Check: Ensure it is a training file (more than 1 class)
    if len(unique_classes) > 1:
        print(f"✅ Loading Training Data: {filename} (Subject ID: {subject_id}, Classes: {unique_classes})")
        
        # ==========================================
        # 4. Data Extraction & Appending
        # ==========================================
        # Strictly pulls the 22 EEG channels, automatically dropping the 3 EOG channels
        X_temp = epochs.get_data(picks='eeg')  
        
        # Create a grouping array filled with the current subject's ID
        group_temp = np.full(len(y_temp), subject_id)
        
        # Append to the temporary memory buckets
        X_all.append(X_temp)
        y_all.append(y_temp)
        groups_all.append(group_temp)
        
    else:
         print(f"  -> ⏭️ Skipping Evaluation Data: {filename} (Classes: {unique_classes})")
         
    # ==========================================
    # 5. Memory Flush
    # ==========================================
    # Explicit garbage collection before loading the next subject
    del epochs

print("\n" + "=" * 60)
print("Consolidating Multi-Subject Tensors...")

# ==========================================
# 6. Final Array Concatenation
# ==========================================
if X_all:
    # Convert lists of arrays into massive continuous NumPy tensors along the epoch axis (axis=0)
    X_all = np.concatenate(X_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    groups_all = np.concatenate(groups_all, axis=0)
    
    # ==========================================
    # 7. Verification Printout
    # ==========================================
    print(f"✅ Final Feature Tensor (X_all) Shape: {X_all.shape}")
    print(f"✅ Final Labels Array (y_all) Shape:   {y_all.shape}")
    print(f"✅ Final Groups Array (groups_all) Shape: {groups_all.shape}")
    print(f"✅ Subjects Loaded into Global Array: {np.unique(groups_all)}")
    
else:
    print("❌ No valid training data found to concatenate.")
    
print("=" * 60)

Starting Multi-Subject Data Loader (LOSO Preparation)...
  -> ⏭️ Skipping Evaluation Data: A01E_clean-epo.fif (Classes: [0])
✅ Loading Training Data: A01T_clean-epo.fif (Subject ID: 1, Classes: [1 2 3 4])
  -> ⏭️ Skipping Evaluation Data: A02E_clean-epo.fif (Classes: [0])
✅ Loading Training Data: A02T_clean-epo.fif (Subject ID: 2, Classes: [1 2 3 4])
  -> ⏭️ Skipping Evaluation Data: A03E_clean-epo.fif (Classes: [0])
✅ Loading Training Data: A03T_clean-epo.fif (Subject ID: 3, Classes: [1 2 3 4])
  -> ⏭️ Skipping Evaluation Data: A04E_clean-epo.fif (Classes: [0])
✅ Loading Training Data: A04T_clean-epo.fif (Subject ID: 4, Classes: [1 2 3 4])
  -> ⏭️ Skipping Evaluation Data: A05E_clean-epo.fif (Classes: [0])
✅ Loading Training Data: A05T_clean-epo.fif (Subject ID: 5, Classes: [1 2 3 4])
  -> ⏭️ Skipping Evaluation Data: A06E_clean-epo.fif (Classes: [0])
✅ Loading Training Data: A06T_clean-epo.fif (Subject ID: 6, Classes: [1 2 3 4])
  -> ⏭️ Skipping Evaluation Data: A07E_clean-epo.fif (C

In [9]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.metrics import accuracy_score

# Import pyRiemann modules for Riemannian Geometry manifold projection
from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

print("Starting Generalized LOSO BCI Pipeline (Riemannian Geometry)...")
print("=" * 60)

# ==========================================
# 1. Define the Riemannian Pipeline
# ==========================================
# Phase 1: Covariances - Converts the (22, 751) EEG trials into (22, 22) Spatial Covariance Matrices
# Phase 2: TangentSpace - Projects the curved Riemannian manifold flat into Euclidean space
# Phase 3: SVM - Draws a linear hyperplane through the flattened, multi-subject data
clf_pipeline = Pipeline([
    ('Covariances', Covariances(estimator='lwf')), # Ledoit-Wolf shrinkage stabilizes the multi-subject matrices
    ('TangentSpace', TangentSpace(metric='riemann')),
    ('SVM', SVC(kernel='linear', C=1.0))
])

# ==========================================
# 2. Configure Leave-One-Subject-Out (LOSO)
# ==========================================
# LOGO uses the groups_all array (containing subject IDs 1-9) to mathematically isolate 
# one full subject per fold, ensuring zero data leakage between training and testing brains.
logo = LeaveOneGroupOut()

fold_accuracies = []
subjects_tested = []

print("Initiating Transfer Learning Cross-Validation (9 Folds)...\n")

# ==========================================
# 3. Execute Cross-Validation Loop
# ==========================================
for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_all, y_all, groups_all)):
    
    # Split the tensors for this specific fold
    X_train, y_train = X_all[train_idx], y_all[train_idx]
    X_test, y_test = X_all[test_idx], y_all[test_idx]
    
    # Identify which subject is currently being held out
    test_subject_id = np.unique(groups_all[test_idx])[0]
    
    # Train the pipeline on the 8 pooled subjects
    clf_pipeline.fit(X_train, y_train)
    
    # Predict on the 1 unseen subject
    y_pred = clf_pipeline.predict(X_test)
    
    # Calculate accuracy
    acc = accuracy_score(y_test, y_pred)
    fold_accuracies.append(acc)
    subjects_tested.append(test_subject_id)
    
    print(f"Fold {fold_idx + 1} | Target Subject: A0{test_subject_id} | Accuracy: {acc * 100:.2f}%")

# ==========================================
# 4. Final System Metrics
# ==========================================
print("\n" + "=" * 60)
print("Pipeline Execution Complete. Final LOSO Metrics:")
print(f"Mean Transfer Accuracy: {np.mean(fold_accuracies) * 100:.2f}% +/- {np.std(fold_accuracies) * 100:.2f}%")
print("=" * 60)

Starting Generalized LOSO BCI Pipeline (Riemannian Geometry)...
Initiating Transfer Learning Cross-Validation (9 Folds)...

Fold 1 | Target Subject: A01 | Accuracy: 45.42%
Fold 2 | Target Subject: A02 | Accuracy: 25.56%
Fold 3 | Target Subject: A03 | Accuracy: 56.30%
Fold 4 | Target Subject: A04 | Accuracy: 31.68%
Fold 5 | Target Subject: A05 | Accuracy: 24.05%
Fold 6 | Target Subject: A06 | Accuracy: 29.22%
Fold 7 | Target Subject: A07 | Accuracy: 26.94%
Fold 8 | Target Subject: A08 | Accuracy: 42.80%
Fold 9 | Target Subject: A09 | Accuracy: 43.88%

Pipeline Execution Complete. Final LOSO Metrics:
Mean Transfer Accuracy: 36.20% +/- 10.58%


In [11]:
import numpy as np
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.model_selection import LeaveOneGroupOut, train_test_split
from sklearn.metrics import accuracy_score

from pyriemann.estimation import Covariances
from pyriemann.tangentspace import TangentSpace

print("Starting Offline Dynamic Calibrated LOSO Pipeline...")
print("=" * 60)

# 1. Define the Riemannian Pipeline
clf_pipeline = Pipeline([
    ('Covariances', Covariances(estimator='lwf')), 
    ('TangentSpace', TangentSpace(metric='riemann')),
    ('SVM', SVC(kernel='linear', C=1.0))
])

logo = LeaveOneGroupOut()
fold_accuracies = []

# 2. Execute Dynamic Calibrated Cross-Validation Loop
for fold_idx, (train_idx, test_idx) in enumerate(logo.split(X_all, y_all, groups_all)):
    
    # Isolate the 8 training subjects and the 1 test subject
    X_train_base, y_train_base = X_all[train_idx], y_all[train_idx]
    X_test_full, y_test_full = X_all[test_idx], y_all[test_idx]
    
    test_subject_id = np.unique(groups_all[test_idx])[0]
    
    # Dynamic Calibration Variables
    shots_per_class = 3
    max_shots = 15      # Hard limit to prevent it from eating the entire test set
    acc = 0.0
    
    print(f"\nTarget Subject: A0{test_subject_id} | Initiating Dynamic Calibration...")
    
    # 3. The Offline Dynamic Calibration Loop
    while acc < 0.65 and shots_per_class <= max_shots:
        total_calib_trials = shots_per_class * 4
        
        # Failsafe in case we ask for more trials than exist in the array
        if total_calib_trials >= len(y_test_full):
            print("  -> ⚠️ Hit maximum available trials for this subject. Breaking early.")
            break
        
        # Stratify ensures we get exactly an equal number of each class
        X_eval, X_calib, y_eval, y_calib = train_test_split(
            X_test_full, y_test_full, 
            test_size=total_calib_trials, 
            stratify=y_test_full,
            random_state=42 # Locked for reproducibility
        )
        
        # Inject the dynamic calibration trials into the main training pool
        X_train_calibrated = np.concatenate((X_train_base, X_calib), axis=0)
        y_train_calibrated = np.concatenate((y_train_base, y_calib), axis=0)
        
        # Train & Evaluate
        clf_pipeline.fit(X_train_calibrated, y_train_calibrated)
        y_pred = clf_pipeline.predict(X_eval)
        
        acc = accuracy_score(y_eval, y_pred)
        
        print(f"  -> {shots_per_class}-Shot Attempt: {acc * 100:.2f}%")
        
        # If it failed to hit 65%, add 2 more shots per class for the next loop iteration
        if acc < 0.65:
            shots_per_class += 2 
            
    fold_accuracies.append(acc)
    print(f"✅ Fold Locked | Final Evaluation Accuracy: {acc * 100:.2f}%")

print("\n" + "=" * 60)
print("Pipeline Execution Complete. Final Dynamic Metrics:")
print(f"Mean Transfer Accuracy: {np.mean(fold_accuracies) * 100:.2f}% +/- {np.std(fold_accuracies) * 100:.2f}%")
print("=" * 60)

Starting Offline Dynamic Calibrated LOSO Pipeline...

Target Subject: A01 | Initiating Dynamic Calibration...
  -> 3-Shot Attempt: 52.11%
  -> 5-Shot Attempt: 55.34%
  -> 7-Shot Attempt: 57.96%
  -> 9-Shot Attempt: 62.45%
  -> 11-Shot Attempt: 62.01%
  -> 13-Shot Attempt: 63.35%
  -> 15-Shot Attempt: 63.85%
✅ Fold Locked | Final Evaluation Accuracy: 63.85%

Target Subject: A02 | Initiating Dynamic Calibration...
  -> 3-Shot Attempt: 26.74%
  -> 5-Shot Attempt: 31.60%
  -> 7-Shot Attempt: 30.58%
  -> 9-Shot Attempt: 32.48%
  -> 11-Shot Attempt: 33.19%
  -> 13-Shot Attempt: 33.94%
  -> 15-Shot Attempt: 33.33%
✅ Fold Locked | Final Evaluation Accuracy: 33.33%

Target Subject: A03 | Initiating Dynamic Calibration...
  -> 3-Shot Attempt: 66.28%
✅ Fold Locked | Final Evaluation Accuracy: 66.28%

Target Subject: A04 | Initiating Dynamic Calibration...
  -> 3-Shot Attempt: 42.40%
  -> 5-Shot Attempt: 42.15%
  -> 7-Shot Attempt: 42.31%
  -> 9-Shot Attempt: 42.92%
  -> 11-Shot Attempt: 44.95%
  

In [12]:
import pybullet as p
import pybullet_data
import time
import numpy as np
from collections import deque

print("Booting PyBullet Physics Engine...")

# 1. Initialize PyBullet GUI
physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)

# 2. Load the Environment and the Robotic Arm
plane_id = p.loadURDF("plane.urdf")
# The Kuka iiwa is a standard 7-DOF robotic arm built into PyBullet
arm_id = p.loadURDF("kuka_iiwa/model.urdf", [0, 0, 0], useFixedBase=True)

# 3. Configure the Majority Vote Buffer
buffer_size = 5
prediction_buffer = deque(maxlen=buffer_size)

print("=" * 60)
print("Starting BCI-to-Robot Imitation Learning Loop...")
print("Mapping Subject 9 EEG data directly to Kuka Arm Actuators.")
print("=" * 60)

# 4. The Continuous Inference Loop
# We iterate through the evaluation dataset currently stored in memory
for i in range(len(X_eval)):
    
    # Grab the next biological 'epoch' (simulating a live EEG stream)
    X_live = np.expand_dims(X_eval[i], axis=0)
    true_intent = y_eval[i]
    
    # AI INFERENCE
    raw_prediction = clf_pipeline.predict(X_live)[0]
    prediction_buffer.append(raw_prediction)
    
    # Wait for the buffer to fill before moving
    if len(prediction_buffer) < buffer_size:
        continue 
        
    # SMOOTHING
    majority_vote = max(set(prediction_buffer), key=prediction_buffer.count)
    
    # ACTUATION MAPPING
    # Joint 0 is the base rotation. Joint 3 is the elbow extension.
    if majority_vote == 1:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=0.5)
    elif majority_vote == 2:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=-0.5)
    elif majority_vote == 3:
        p.setJointMotorControl2(arm_id, jointIndex=3, controlMode=p.VELOCITY_CONTROL, targetVelocity=0.5)
    elif majority_vote == 4:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=0)
        p.setJointMotorControl2(arm_id, jointIndex=3, controlMode=p.VELOCITY_CONTROL, targetVelocity=0)
    
    # Print telemetry to the console
    if i % 5 == 0:
        match = "✅" if majority_vote == true_intent else "❌"
        print(f"Step {i:03d} | True Intent: {true_intent} | AI Command: {majority_vote} {match}")

    # Step the physics engine enough times to visually see the movement 
    # (Simulates the physical time it takes for the robot to execute the thought)
    for _ in range(60):
        p.stepSimulation()
        time.sleep(1./240.)

print("Simulation Complete.")
time.sleep(2)
p.disconnect()

Booting PyBullet Physics Engine...
Starting BCI-to-Robot Imitation Learning Loop...
Mapping Subject 9 EEG data directly to Kuka Arm Actuators.
Step 005 | True Intent: 3 | AI Command: 2 ❌
Step 010 | True Intent: 3 | AI Command: 1 ❌
Step 015 | True Intent: 3 | AI Command: 1 ❌
Step 020 | True Intent: 3 | AI Command: 1 ❌
Step 025 | True Intent: 2 | AI Command: 4 ❌
Step 030 | True Intent: 3 | AI Command: 4 ❌
Step 035 | True Intent: 4 | AI Command: 4 ✅
Step 040 | True Intent: 4 | AI Command: 1 ❌
Step 045 | True Intent: 2 | AI Command: 1 ❌
Step 050 | True Intent: 4 | AI Command: 4 ✅
Step 055 | True Intent: 4 | AI Command: 1 ❌
Step 060 | True Intent: 4 | AI Command: 3 ❌
Step 065 | True Intent: 1 | AI Command: 1 ✅
Step 070 | True Intent: 2 | AI Command: 1 ❌
Step 075 | True Intent: 3 | AI Command: 3 ✅
Step 080 | True Intent: 4 | AI Command: 4 ✅
Step 085 | True Intent: 3 | AI Command: 4 ❌
Step 090 | True Intent: 2 | AI Command: 1 ❌
Step 095 | True Intent: 1 | AI Command: 1 ✅
Step 100 | True Inten

In [13]:
import pybullet as p
import time

print("Attempting to open PyBullet GUI...")
p.connect(p.GUI)
time.sleep(5)
p.disconnect()
print("Test complete.")

Attempting to open PyBullet GUI...
Test complete.


In [20]:
import pybullet as p
import pybullet_data
import time
import numpy as np
from collections import deque

print("Booting PyBullet Physics Engine...")

# 1. Clean up old instances and Initialize
try:
    p.disconnect() 
except:
    pass

physicsClient = p.connect(p.GUI)
p.setAdditionalSearchPath(pybullet_data.getDataPath())
p.setGravity(0, 0, -9.81)

plane_id = p.loadURDF("plane.urdf")
arm_id = p.loadURDF("kuka_iiwa/model.urdf", [0, 0, 0], useFixedBase=True)

# ==========================================
# NEW: CAMERA & UI SETUP GOES EXACTLY HERE
# ==========================================
p.resetDebugVisualizerCamera(cameraDistance=1.5, cameraYaw=45, cameraPitch=-30, cameraTargetPosition=[0, 0, 0.5])
p.configureDebugVisualizer(p.COV_ENABLE_GUI, 0)
# ==========================================

# 2. Sort the Evaluation Data (Simulating sustained continuous intention)
# This groups all Class 1s, then Class 2s, etc.
sorted_indices = np.argsort(y_eval)
X_sorted = X_eval[sorted_indices]
y_sorted = y_eval[sorted_indices]

# 3. Configure the Buffer
buffer_size = 5
prediction_buffer = deque(maxlen=buffer_size)

print("=" * 60)
print(f"Starting Filtered Imitation Learning Loop (Buffer: {buffer_size})...")
print("Data sorted to simulate continuous sustained human intent.")
print("=" * 60)

# 4. The Continuous Buffered Inference Loop
for i in range(len(X_sorted)):
    
    X_live = np.expand_dims(X_sorted[i], axis=0)
    true_intent = y_sorted[i]
    
    # AI INFERENCE
    raw_prediction = clf_pipeline.predict(X_live)[0]
    prediction_buffer.append(raw_prediction)
    
    # Wait for the buffer to fill before the robot is allowed to move
    if len(prediction_buffer) < buffer_size:
        continue
        
    # COMMAND SMOOTHING
    majority_vote = max(set(prediction_buffer), key=prediction_buffer.count)
    
    # ACTUATION MAPPING
    if majority_vote == 1:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=0.8)
    elif majority_vote == 2:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=-0.8)
    elif majority_vote == 3:
        p.setJointMotorControl2(arm_id, jointIndex=3, controlMode=p.VELOCITY_CONTROL, targetVelocity=0.8)
    elif majority_vote == 4:
        p.setJointMotorControl2(arm_id, jointIndex=0, controlMode=p.VELOCITY_CONTROL, targetVelocity=0)
        p.setJointMotorControl2(arm_id, jointIndex=3, controlMode=p.VELOCITY_CONTROL, targetVelocity=0)
    
    match = "✅" if majority_vote == true_intent else "❌"
    
    # Only print every 5th step to keep the console clean
    if i % 5 == 0:
        print(f"Block {i:03d} | True Intent: {true_intent} | Filtered Command: {majority_vote} {match}")

    # Step the simulation forward
    for _ in range(30):
        p.stepSimulation()
        time.sleep(1./240.)

print("\n" + "=" * 60)
print("Simulation Complete.")
print("=" * 60)

time.sleep(2)
p.disconnect()

Booting PyBullet Physics Engine...
Starting Filtered Imitation Learning Loop (Buffer: 5)...
Data sorted to simulate continuous sustained human intent.
Block 005 | True Intent: 1 | Filtered Command: 1 ✅
Block 010 | True Intent: 1 | Filtered Command: 1 ✅
Block 015 | True Intent: 1 | Filtered Command: 1 ✅
Block 020 | True Intent: 1 | Filtered Command: 1 ✅
Block 025 | True Intent: 1 | Filtered Command: 1 ✅
Block 030 | True Intent: 1 | Filtered Command: 1 ✅
Block 035 | True Intent: 1 | Filtered Command: 1 ✅
Block 040 | True Intent: 2 | Filtered Command: 1 ❌
Block 045 | True Intent: 2 | Filtered Command: 2 ✅
Block 050 | True Intent: 2 | Filtered Command: 2 ✅
Block 055 | True Intent: 2 | Filtered Command: 2 ✅
Block 060 | True Intent: 2 | Filtered Command: 2 ✅
Block 065 | True Intent: 2 | Filtered Command: 1 ❌
Block 070 | True Intent: 2 | Filtered Command: 2 ✅
Block 075 | True Intent: 2 | Filtered Command: 4 ❌
Block 080 | True Intent: 2 | Filtered Command: 1 ❌
Block 085 | True Intent: 2 | Filt

visualisation

In [ ]:
import os
import random
import mne
import matplotlib.pyplot as plt

# Suppress warnings to keep console output clean
mne.set_log_level('ERROR')

print("🔭 Initializing Hybrid Static Stack QA Visualizer...")

# ==========================================
# Step 1. Initialization & Target Lock
# ==========================================
epoched_dir = "epoched_eeg_data"

# List all '_clean-epo.fif' files that belong to a Training session
epoched_files = [f for f in os.listdir(epoched_dir) if f.endswith('_clean-epo.fif') and 'T' in f]

if not epoched_files:
    print(f"❌ No training epoch files found in '{epoched_dir}'.")
else:
    # Randomly select one file and strip the suffix to isolate the base_name
    target_epo_file = random.choice(epoched_files)
    base_name = target_epo_file.split('_')[0] 
    print(f"✅ Random Target Locked: Subject/Session {base_name}")

    print("Loading the 4 stages of the pipeline into memory...")
    
    # ==========================================
    # Step 2. Stage 1 - Raw Ingestion
    # ==========================================
    # Load [base_name].gdf from the raweegdata folder
    raw_filepath = os.path.join("raweegdata", f"{base_name}.gdf")
    stage1_raw = mne.io.read_raw_gdf(raw_filepath, preload=True)
    
    # Explicitly map the EOG channels so the plotting engine scales vertical voltages correctly
    stage1_raw.set_channel_types({'EOG-left': 'eog', 'EOG-central': 'eog', 'EOG-right': 'eog'})
    
    # ==========================================
    # Step 3. Stage 2 - The Pipeline Clone (Script 1 Logic)
    # ==========================================
    print("  -> Recreating intermediate filtering stage with accurate double-FIR...")
    
    # Execute .copy() to isolate the memory
    stage2_filtered = stage1_raw.copy()
    
    # Apply high-pass and SMR bandpass filters
    stage2_filtered.filter(l_freq=1.0, h_freq=None, fir_design='firwin') 
    stage2_filtered.filter(l_freq=8.0, h_freq=30.0, fir_design='firwin') 
    
    # ==========================================
    # Step 4. Stage 3 & 4 - Clean & Epoched Loading
    # ==========================================
    # Load the pre-saved _clean.fif file from the ICA phase
    clean_filepath = os.path.join("final_clean_eeg", f"{base_name}_clean.fif")
    stage3_clean = mne.io.read_raw_fif(clean_filepath, preload=True)
    
    # Load the pre-saved _clean-epo.fif tensor from the Epoching phase
    epo_filepath = os.path.join(epoched_dir, target_epo_file)
    stage4_epochs = mne.read_epochs(epo_filepath, preload=True)

    # ==========================================
    # Step 5. The "God's Eye" Render (Script 2 Logic)
    # ==========================================
    print("\nDrawing High-Resolution Stack...")
    
    # Hardcode time slicing to exactly 5.0s to 15.0s
    tmin, tmax = 5.0, 15.0
    
    # Initialize a large 15x20 Matplotlib figure with 4 stacked rows
    fig, axes = plt.subplots(4, 1, figsize=(15, 20), sharex=False)
    fig.suptitle(f"God's Eye Pipeline QA: Subject {base_name}", fontsize=16, fontweight='bold')

    # Render Stage 1, 2, and 3 into the first three rows
    stage1_raw.plot(axes=axes[0], show=False, duration=tmax-tmin, start=tmin)
    axes[0].set_title("Stage 1: Raw Amplified Data (Look for EOG Spikes)")

    stage2_filtered.plot(axes=axes[1], show=False, duration=tmax-tmin, start=tmin)
    axes[1].set_title("Stage 2: FIR Filtered (1.0Hz HP + 8-30Hz BP) - Baseline stabilized, blinks remain")

    stage3_clean.plot(axes=axes[2], show=False, duration=tmax-tmin, start=tmin)
    axes[2].set_title("Stage 3: ICA Cleaned - Blinks mathematically erased")

    # Render 5 slices of the Stage 4 tensor into the bottom row
    stage4_epochs.plot(axes=axes[3], show=False, n_epochs=5)
    axes[3].set_title("Stage 4: Epoched Tensors (Ready for CSP)")

    plt.tight_layout(rect=[0, 0.03, 1, 0.97])
    
    # Halt script execution and display the image
    plt.show()

    # ==========================================
    # Step 6. Garbage Collection
    # ==========================================
    # Once the user closes the Matplotlib window, delete the heavy EEG arrays from the RAM buffer
    del stage1_raw, stage2_filtered, stage3_clean, stage4_epochs
    print("✅ Visualization complete. RAM flushed. Files secured.")

In [ ]:
import os
import random
import mne
import matplotlib.pyplot as plt

# Suppress warnings to keep console output clean
mne.set_log_level('ERROR')

print("🔭 Initializing Hybrid Static Stack QA Visualizer...")

# ==========================================
# Step 1. Initialization & Target Lock
# ==========================================
epoched_dir = "epoched_eeg_data"

# List all '_clean-epo.fif' files that belong to a Training session
epoched_files = [f for f in os.listdir(epoched_dir) if f.endswith('_clean-epo.fif') and 'T' in f]

if not epoched_files:
    print(f"❌ No training epoch files found in '{epoched_dir}'.")
else:
    # Randomly select one file and strip the suffix to isolate the base_name
    target_epo_file = random.choice(epoched_files)
    base_name = target_epo_file.split('_')[0] 
    print(f"✅ Random Target Locked: Subject/Session {base_name}")

    print("Loading the 4 stages of the pipeline into memory...")
    
    # ==========================================
    # Step 2. Stage 1 - Raw Ingestion
    # ==========================================
    # Load [base_name].gdf from the raweegdata folder
    raw_filepath = os.path.join("raweegdata", f"{base_name}.gdf")
    stage1_raw = mne.io.read_raw_gdf(raw_filepath, preload=True)
    
    # Explicitly map the EOG channels so the plotting engine scales vertical voltages correctly
    stage1_raw.set_channel_types({'EOG-left': 'eog', 'EOG-central': 'eog', 'EOG-right': 'eog'})
    
    # ==========================================
    # Step 3. Stage 2 - The Pipeline Clone (Script 1 Logic)
    # ==========================================
    print("  -> Recreating intermediate filtering stage with accurate double-FIR...")
    
    # Execute .copy() to isolate the memory
    stage2_filtered = stage1_raw.copy()
    
    # Apply high-pass and SMR bandpass filters
    stage2_filtered.filter(l_freq=1.0, h_freq=None, fir_design='firwin') 
    stage2_filtered.filter(l_freq=8.0, h_freq=30.0, fir_design='firwin') 
    
    # ==========================================
    # Step 4. Stage 3 & 4 - Clean & Epoched Loading
    # ==========================================
    # Load the pre-saved _clean.fif file from the ICA phase
    clean_filepath = os.path.join("final_clean_eeg", f"{base_name}_clean.fif")
    stage3_clean = mne.io.read_raw_fif(clean_filepath, preload=True)
    
    # Load the pre-saved _clean-epo.fif tensor from the Epoching phase
    epo_filepath = os.path.join(epoched_dir, target_epo_file)
    stage4_epochs = mne.read_epochs(epo_filepath, preload=True)

    #print("\nDrawing High-Resolution Stack...")

tmin, tmax = 5.0, 15.0
fig, axes = plt.subplots(4, 1, figsize=(15, 20))
fig.suptitle(f"God's Eye Pipeline QA: Subject {base_name}", fontsize=16, fontweight='bold')

# Helper to plot MNE data on Matplotlib axes
def plot_raw_on_axes(raw_obj, ax, title, start, duration):
    # Extract data as array: (channels, time)
    data, times = raw_obj.get_data(start=raw_obj.time_as_index(start)[0], 
                                   stop=raw_obj.time_as_index(start + duration)[0], 
                                   return_times=True)
    # Plot first 10 channels for clarity
    for i in range(min(10, data.shape[0])):
        ax.plot(times, data[i, :] + (i * 20e-6), color='k', linewidth=0.5)
    ax.set_title(title)
    ax.set_yticks([]) # Hide channel labels to keep it clean

# Render Stages 1, 2, and 3
plot_raw_on_axes(stage1_raw, axes[0], "Stage 1: Raw Amplified Data", tmin, tmax-tmin)
plot_raw_on_axes(stage2_filtered, axes[1], "Stage 2: FIR Filtered (8-30Hz)", tmin, tmax-tmin)
plot_raw_on_axes(stage3_clean, axes[2], "Stage 3: ICA Cleaned", tmin, tmax-tmin)

# Plot 4: Epoched (Using MNE's built-in plotting for epochs is fine because it's a different class)
# We select the first 5 epochs to visualize
stage4_epochs.plot(n_epochs=5, picks='eeg', show=False) 
# Note: Since stage4_epochs.plot() still doesn't accept 'axes', we let it open its own 
# window or simply skip it if the stack is the priority. 
# To keep it in the stack, we plot the first trial average:
evoked = stage4_epochs.average()
evoked.plot(axes=axes[3], show=False)
axes[3].set_title("Stage 4: Epoched Evoked Response")

plt.tight_layout(rect=[0, 0.03, 1, 0.97])
plt.show()

# 4. Strict Garbage Collection
del stage1_raw, stage2_filtered, stage3_clean, stage4_epochs